# ToyGPT — Google Colab 실습

**GPT를 처음부터 구현하는 강의용 예제**

| 단계 | 스크립트 | 설명 |
|------|---------|------|
| 1 | Section 3 | 사전 학습 — 대량 텍스트로 다음 문자 예측 |
| 2 | Section 4 | 파인튜닝 — Q&A 형식 학습 |
| 3 | Section 5 | 대화 — 실시간 스트리밍 생성 |

> **실행 순서**: Section 2에서 언어를 선택한 후, 셀을 위에서 아래로 순서대로 실행하세요.
> 메뉴 **런타임 → 모두 실행** 으로 한 번에 실행할 수 있습니다.

## Section 0: 환경 설정

In [ ]:
# 패키지 설치 (Colab에 이미 설치된 경우 빠르게 완료됨)
!pip install torch numpy matplotlib -q
print("설치 완료!")

In [ ]:
import torch

# GPU 상태 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU 사용 가능: {gpu_name} ({gpu_mem:.1f} GB)")
    print("  → 큰 모델(d_model=128, layers=4)로 자동 설정됩니다.")
else:
    print("GPU 없음 — CPU 모드")
    print("  런타임 → 런타임 유형 변경 → GPU 를 선택하면 빠릅니다.")
    print("  → 작은 모델(d_model=64, layers=2)로 자동 설정됩니다.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

In [ ]:
# [선택사항] Google Drive 마운트 — 체크포인트를 세션 종료 후에도 보존하려면 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/ToyGPT_checkpoints'

# Drive 미사용 시 로컬 경로 사용 (세션 종료 시 삭제됨)
import os
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"체크포인트 저장 경로: {CHECKPOINT_DIR}/")

## Section 1: 공통 모듈 정의

`%%writefile` 매직 커맨드가 `.py` 파일을 현재 디렉토리에 생성합니다.  
이후 셀에서 `from tokenizer import CharTokenizer` 처럼 임포트하여 사용합니다.

In [ ]:
%%writefile tokenizer.py
"""
tokenizer.py — 문자 단위 토크나이저 (Character-level Tokenizer)

핵심 개념:
  - 텍스트의 모든 고유 문자를 정수로 매핑합니다.
  - 예: 'A' → 0, 'B' → 1, ... (학습 코퍼스 기준)
  - vocab_size가 매우 작아(~70) 모델 구조를 직관적으로 이해할 수 있습니다.
"""

import json


class CharTokenizer:
    """
    문자 단위 토크나이저.

    사용 예시:
        tokenizer = CharTokenizer("hello world")
        ids = tokenizer.encode("hello")   # [7, 4, 11, 11, 14]
        text = tokenizer.decode(ids)      # "hello"
    """

    def __init__(self, text: str):
        """
        텍스트 코퍼스에서 어휘(vocab)를 자동으로 구축합니다.

        Args:
            text: 어휘를 추출할 원본 텍스트
        """
        # 고유 문자를 정렬하여 재현 가능한 vocab 생성
        chars = sorted(set(text))
        self.vocab = {ch: i for i, ch in enumerate(chars)}       # char → id
        self.inv_vocab = {i: ch for ch, i in self.vocab.items()} # id → char

    @property
    def vocab_size(self) -> int:
        """어휘 크기 (고유 문자 수)"""
        return len(self.vocab)

    def encode(self, text: str) -> list:
        """
        문자열을 정수 리스트로 변환합니다.

        Args:
            text: 인코딩할 문자열
        Returns:
            정수 ID 리스트
        """
        return [self.vocab[ch] for ch in text if ch in self.vocab]

    def decode(self, ids: list) -> str:
        """
        정수 리스트를 문자열로 변환합니다.

        Args:
            ids: 정수 ID 리스트
        Returns:
            복원된 문자열
        """
        return ''.join(self.inv_vocab.get(i, '') for i in ids)

    def save(self, path: str):
        """
        토크나이저를 JSON 파일로 저장합니다.
        모델 체크포인트와 함께 보관하여 추후 로드 시 사용합니다.
        """
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)
        print(f"토크나이저 저장됨: {path}  (vocab_size={self.vocab_size})")

    @classmethod
    def load(cls, path: str) -> 'CharTokenizer':
        """
        저장된 JSON 파일에서 토크나이저를 복원합니다.

        Args:
            path: tokenizer.save()로 저장한 JSON 경로
        Returns:
            복원된 CharTokenizer 인스턴스
        """
        # 빈 인스턴스를 만들고 vocab을 직접 주입합니다.
        tokenizer = cls.__new__(cls)
        with open(path, 'r', encoding='utf-8') as f:
            tokenizer.vocab = json.load(f)
        tokenizer.inv_vocab = {v: k for k, v in tokenizer.vocab.items()}
        return tokenizer


# ──────────────────────────────────────────────
# 실행 시 간단한 동작 확인
# ──────────────────────────────────────────────
if __name__ == '__main__':
    sample_text = "Hello, World! 안녕하세요."
    tok = CharTokenizer(sample_text)

    print(f"vocab_size : {tok.vocab_size}")
    print(f"vocab      : {tok.vocab}")

    ids = tok.encode("Hello")
    print(f"encode('Hello') → {ids}")
    print(f"decode({ids})   → '{tok.decode(ids)}'")


In [ ]:
%%writefile model.py
"""
model.py — ToyGPT: 강의용 미니 GPT 모델

구조 (GPT-2 논문과 동일한 아키텍처):

  입력 토큰 ids
      ↓
  Token Embedding  +  Position Embedding
      ↓
  TransformerBlock × n_layers
  (각 블록 = CausalSelfAttention + FeedForward + LayerNorm × 2)
      ↓
  LayerNorm (최종)
      ↓
  LM Head (Linear, weight tying)
      ↓
  logits → (softmax) → 다음 토큰 확률

[핵심 개념]
  - Causal Mask: 미래 토큰을 볼 수 없게 하는 삼각형 마스크
  - Weight Tying: Embedding과 LM Head의 가중치를 공유 → 파라미터 절약
  - Autoregressive: 토큰을 하나씩 생성 (generate 메서드 참고)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════
# 1. Causal Self-Attention (인과적 자기 주의)
# ══════════════════════════════════════════════════════════════

class CausalSelfAttention(nn.Module):
    """
    마스킹된 멀티헤드 셀프 어텐션.

    "Attention is All You Need" (Vaswani et al., 2017)의 핵심:
        Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V

    Causal(인과적): 현재 위치 t는 t 이전 토큰만 볼 수 있습니다.
    이를 위해 미래 위치를 -inf로 마스킹합니다.
    """

    def __init__(self, d_model: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0, "d_model은 n_heads로 나누어져야 합니다."

        self.n_heads = n_heads
        self.d_head = d_model // n_heads  # 헤드당 차원

        # Q, K, V를 한 번에 계산하는 행렬 (효율적)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        # 멀티헤드 결과를 합치는 출력 행렬
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Causal Mask: 하삼각행렬 (미래 토큰 → -inf)
        # register_buffer: 학습 파라미터가 아니지만 체크포인트에 저장됨
        mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer('mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # (배치, 시퀀스 길이, d_model)

        # ── Q, K, V 계산 ──────────────────────────────────────
        qkv = self.qkv_proj(x)                          # (B, T, 3*C)
        Q, K, V = qkv.split(C, dim=-1)                  # 각각 (B, T, C)

        # 멀티헤드: (B, T, C) → (B, n_heads, T, d_head)
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        # ── Scaled Dot-Product Attention ───────────────────────
        # 스케일링: d_head의 제곱근으로 나눠 gradient 안정화
        scale = math.sqrt(self.d_head)
        scores = (Q @ K.transpose(-2, -1)) / scale      # (B, H, T, T)

        # Causal Masking: 미래 위치를 매우 작은 값으로 → softmax 후 0이 됨
        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float('-inf')
        )

        attn_weights = F.softmax(scores, dim=-1)         # (B, H, T, T)
        attn_weights = self.attn_dropout(attn_weights)

        # ── V와 결합 ────────────────────────────────────────────
        out = attn_weights @ V                           # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # 헤드 합치기
        out = self.resid_dropout(self.out_proj(out))
        return out


# ══════════════════════════════════════════════════════════════
# 2. Feed-Forward Network (피드포워드 네트워크)
# ══════════════════════════════════════════════════════════════

class FeedForward(nn.Module):
    """
    2층 MLP with GELU 활성화.

    각 토큰을 독립적으로 처리합니다 (position-wise).
    d_ff는 보통 d_model의 4배 (논문 관례).
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),          # ReLU보다 부드러운 활성화 함수 (GPT-2 사용)
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ══════════════════════════════════════════════════════════════
# 3. Transformer Block (트랜스포머 블록)
# ══════════════════════════════════════════════════════════════

class TransformerBlock(nn.Module):
    """
    Pre-LayerNorm Transformer 블록 (GPT-2 스타일).

    Pre-LN 구조 (GPT-2):         Post-LN 구조 (원논문):
        x → LN → Attn → + x         x → Attn → + x → LN
        x → LN → FF  → + x          x → FF  → + x → LN

    Pre-LN이 학습이 더 안정적입니다.
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 context_len: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, context_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Residual Connection (잔차 연결): 그래디언트 흐름 보장
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


# ══════════════════════════════════════════════════════════════
# 4. ToyGPT — 전체 모델
# ══════════════════════════════════════════════════════════════

class ToyGPT(nn.Module):
    """
    강의용 미니 GPT 언어 모델.

    기본 설정 (CPU에서 2-3분 학습):
        vocab_size  : ~70 (문자 단위)
        context_len : 128
        d_model     : 64
        n_layers    : 2
        n_heads     : 4
        d_ff        : 256
        → 총 파라미터: ~200K
    """

    def __init__(
        self,
        vocab_size: int,
        context_len: int = 128,
        d_model: int = 64,
        n_layers: int = 2,
        n_heads: int = 4,
        d_ff: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.context_len = context_len

        # ── 임베딩 레이어 ───────────────────────────────────────
        # 토큰 임베딩: 각 정수 ID → d_model 차원 벡터
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # 위치 임베딩: 각 위치(0~context_len) → d_model 차원 벡터
        self.position_embedding = nn.Embedding(context_len, d_model)

        self.drop = nn.Dropout(dropout)

        # ── Transformer 블록 스택 ────────────────────────────────
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, context_len, dropout)
            for _ in range(n_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)

        # ── LM Head (언어 모델 헤드) ────────────────────────────
        # d_model → vocab_size: 각 위치에서 다음 토큰의 확률 예측
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight Tying: 입력 임베딩과 출력 행렬을 공유
        # → 같은 토큰이 입력과 출력에서 유사한 표현을 가져야 한다는 직관
        self.lm_head.weight = self.token_embedding.weight

        # 가중치 초기화 (GPT-2 스타일)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """가중치 초기화: Linear → 표준편차 0.02 정규분포"""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def count_parameters(self) -> int:
        """학습 가능한 파라미터 수를 반환합니다."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor = None,
    ):
        """
        순전파.

        Args:
            idx    : (B, T) — 입력 토큰 ID
            targets: (B, T) — 정답 토큰 ID (학습 시), 없으면 None

        Returns:
            logits : (B, T, vocab_size) — 각 위치의 다음 토큰 로짓
            loss   : CrossEntropy 손실 (targets가 None이면 None)
        """
        B, T = idx.shape
        assert T <= self.context_len, \
            f"시퀀스 길이 {T}가 context_len {self.context_len}을 초과합니다."

        # 위치 ID: [0, 1, 2, ..., T-1]
        positions = torch.arange(T, device=idx.device)

        # 토큰 임베딩 + 위치 임베딩 합산
        x = self.drop(
            self.token_embedding(idx) + self.position_embedding(positions)
        )  # (B, T, d_model)

        # Transformer 블록 순차 통과
        for block in self.blocks:
            x = block(x)

        x = self.ln_final(x)                            # (B, T, d_model)
        logits = self.lm_head(x)                         # (B, T, vocab_size)

        # 손실 계산 (학습 모드)
        loss = None
        if targets is not None:
            # (B, T, vocab_size) → (B*T, vocab_size) 로 reshape 후 CrossEntropy
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int = 40,
    ) -> torch.Tensor:
        """
        자기회귀적(autoregressive) 토큰 생성.

        핵심: 토큰을 하나씩 생성하며, 매번 새로운 forward pass를 수행합니다.
        이것이 GPT의 실제 동작 방식입니다.

        Args:
            idx           : (1, T) — 시드(seed) 토큰
            max_new_tokens: 생성할 최대 토큰 수
            temperature   : 높을수록 다양한 출력 (기본 1.0)
            top_k         : 상위 k개 토큰 중에서만 샘플링

        Returns:
            (1, T + max_new_tokens) — 확장된 토큰 시퀀스
        """
        self.eval()
        for _ in range(max_new_tokens):
            # context_len 초과 시 오른쪽 잘라냄
            idx_cond = idx[:, -self.context_len:]

            # 순전파: 마지막 위치의 logits만 사용
            logits, _ = self(idx_cond)
            next_logits = logits[:, -1, :]              # (1, vocab_size)

            # Temperature 스케일링
            next_logits = next_logits / temperature

            # Top-k 필터링: 상위 k개 외의 토큰을 -inf로 설정
            if top_k is not None:
                top_vals, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                threshold = top_vals[:, -1].unsqueeze(-1)
                next_logits = next_logits.masked_fill(next_logits < threshold, float('-inf'))

            # 확률 분포에서 샘플링
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)

            # 생성된 토큰을 시퀀스에 추가
            idx = torch.cat([idx, next_token], dim=1)

        return idx


# ──────────────────────────────────────────────
# 실행 시 모델 구조 및 파라미터 수 출력
# ──────────────────────────────────────────────
if __name__ == '__main__':
    model = ToyGPT(vocab_size=70)
    print(model)
    print(f"\n총 파라미터 수: {model.count_parameters():,}")

    # 더미 입력으로 동작 확인
    dummy_input = torch.randint(0, 70, (2, 32))   # (batch=2, seq=32)
    dummy_target = torch.randint(0, 70, (2, 32))
    logits, loss = model(dummy_input, targets=dummy_target)
    print(f"logits shape : {logits.shape}")        # (2, 32, 70)
    print(f"loss         : {loss.item():.4f}")


In [ ]:
%%writefile data.py
"""
data.py — 데이터셋 헬퍼

[핵심 개념] 언어 모델 학습 데이터:
  - 입력(x): 토큰 [t0, t1, t2, ..., t_{T-1}]
  - 정답(y): 토큰 [t1, t2, t3, ..., t_T]     ← x를 한 칸 오른쪽 shift
  - 즉, "이전 토큰들이 주어졌을 때 다음 토큰 예측"이 목표입니다.
"""

import os
import urllib.request
import torch
from torch.utils.data import Dataset


# ══════════════════════════════════════════════════════════════
# 1. Tiny Shakespeare 데이터 로드
# ══════════════════════════════════════════════════════════════

# Andrej Karpathy의 char-rnn에서 사용한 셰익스피어 데이터
_SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/"
    "master/data/tinyshakespeare/input.txt"
)

# 다운로드 실패 시 사용할 백업 텍스트 (셰익스피어 소네트 일부)
_FALLBACK_TEXT = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.

Second Citizen:
Would you proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very well; and could be content to give him good
report fort, but that he pays himself with being proud.

Second Citizen:
Nay, but speak not maliciously.

First Citizen:
I say unto you, what he hath done famously, he did
it to that end: though soft-conscienced men can be
content to say it was for his country he did it to
please his mother and to be partly proud; which he
is, even till the altitude of his virtue.

Second Citizen:
What he cannot help in his nature, you account a
vice in him. You must in no way say he is covetous.

First Citizen:
If I must not, I need not be barren of accusations;
he hath faults, with surplus, to tire in repetition.
What shouts are these? The other side o' the city
is risen: why stay we prating here? to the Capitol!

All:
Come, come.

First Citizen:
Soft! who comes here?

Second Citizen:
Worthy Menenius Agrippa; one that hath always loved
the people.

First Citizen:
He's one honest enough: would all the rest were so!

Menenius:
What work's, my countrymen, in hand? where go you
With bats and clubs? The matter? speak, I pray you.

First Citizen:
Our business is not unknown to the senate; they have
had inkling this fortnight what we intend to do,
which now we'll show 'em in deeds. They say poor
suitors have strong breaths: they shall know we
have strong arms too.
""" * 20  # 반복하여 충분한 학습 데이터 확보


def get_shakespeare(cache_path: str = "shakespeare.txt") -> str:
    """
    Tiny Shakespeare 텍스트를 반환합니다.

    - 캐시 파일이 있으면 파일에서 로드합니다.
    - 없으면 인터넷에서 다운로드합니다.
    - 다운로드 실패 시 내장 백업 텍스트를 사용합니다.

    Args:
        cache_path: 캐시 파일 경로 (기본: 'shakespeare.txt')
    Returns:
        셰익스피어 텍스트 문자열
    """
    if os.path.exists(cache_path):
        print(f"캐시에서 로드: {cache_path}")
        with open(cache_path, 'r', encoding='utf-8') as f:
            return f.read()

    print(f"다운로드 중: {_SHAKESPEARE_URL}")
    try:
        urllib.request.urlretrieve(_SHAKESPEARE_URL, cache_path)
        print(f"다운로드 완료: {cache_path}")
        with open(cache_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"다운로드 실패: {e}")
        print("내장 백업 텍스트를 사용합니다.")
        # 백업 텍스트를 캐시로 저장
        with open(cache_path, 'w', encoding='utf-8') as f:
            f.write(_FALLBACK_TEXT)
        return _FALLBACK_TEXT


# ══════════════════════════════════════════════════════════════
# 2. TextDataset — PyTorch 데이터셋
# ══════════════════════════════════════════════════════════════

class TextDataset(Dataset):
    """
    슬라이딩 윈도우 방식의 텍스트 데이터셋.

    예시 (context_len=4):
        토큰 시퀀스: [10, 20, 30, 40, 50, 60]
        인덱스 0:  x=[10,20,30,40], y=[20,30,40,50]
        인덱스 1:  x=[20,30,40,50], y=[30,40,50,60]
        ...

    Args:
        token_ids  : 정수 토큰 ID 리스트
        context_len: 하나의 샘플에서 사용할 시퀀스 길이
    """

    def __init__(self, token_ids: list, context_len: int):
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.context_len = context_len

    def __len__(self) -> int:
        # 마지막 context_len 개 위치는 y를 만들 수 없으므로 제외
        return len(self.data) - self.context_len

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.context_len]
        y = self.data[idx + 1 : idx + self.context_len + 1]
        return x, y


# ──────────────────────────────────────────────
# 실행 시 동작 확인
# ──────────────────────────────────────────────
if __name__ == '__main__':
    text = get_shakespeare()
    print(f"텍스트 길이: {len(text):,} 문자")
    print(f"처음 100자:\n{text[:100]}")

    # 간단한 데이터셋 테스트
    from tokenizer import CharTokenizer
    tok = CharTokenizer(text)
    ids = tok.encode(text[:1000])
    ds = TextDataset(ids, context_len=32)
    x, y = ds[0]
    print(f"\n데이터셋 크기: {len(ds)}")
    print(f"x[:10]: {x[:10].tolist()}")
    print(f"y[:10]: {y[:10].tolist()}")
    print(f"→ x[0]='{tok.decode([x[0].item()])}', y[0]='{tok.decode([y[0].item()])}'")


## Section 2: 언어 선택

아래 드롭다운에서 학습 언어를 선택하세요.  
이후 모든 셀(Section 3~5)이 선택된 언어를 사용합니다.

In [ ]:
LANGUAGE = "한국어 (전래동화)"  # @param ["영어 (Shakespeare)", "한국어 (전래동화)"]

IS_KOREAN = LANGUAGE.startswith("한국어")
print(f"선택된 언어: {LANGUAGE}")
print(f"IS_KOREAN   = {IS_KOREAN}")

## Section 3: 사전 학습 (Pre-training)

| 항목 | 영어 모델 | 한국어 모델 (CPU) | 한국어 모델 (GPU) |
|------|---------|-----------------|------------------|
| 학습 데이터 | Tiny Shakespeare (~1MB) | 전래동화 반복 | 전래동화 반복 |
| vocab_size | ~70 | ~470 | ~470 |
| d_model | 64 | 64 | 128 |
| n_layers | 2 | 2 | 4 |
| 소요 시간 | CPU 2~3분 | CPU 2~3분 | GPU 1분 이내 |

**핵심 개념**: 자기지도학습(Self-supervised) — 다음 문자를 예측하는 것만으로 언어를 이해합니다.

In [ ]:
import os, time, torch, sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline
from torch.utils.data import DataLoader, random_split
from tokenizer import CharTokenizer
from model import ToyGPT
from data import TextDataset, get_shakespeare

torch.manual_seed(42)

# ── 하이퍼파라미터 ──
CONTEXT_LEN   = 128
LEARNING_RATE = 3e-4
MAX_EPOCHS    = 5
VAL_RATIO     = 0.1
DROPOUT       = 0.1

# ── GPU 감지 및 모델 크기 자동 설정 ──
if torch.cuda.is_available():
    device     = torch.device('cuda')
    model_cfg  = dict(d_model=128, n_layers=4, n_heads=8, d_ff=512)
    batch_size = 128
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device     = torch.device('cpu')
    model_cfg  = dict(d_model=64, n_layers=2, n_heads=4, d_ff=256)
    batch_size = 64
    print("CPU 모드")

# ── 학습 데이터 준비 ──
print('[STEP 1] 데이터 준비 중...')

if IS_KOREAN:
    # 한국어: 전래동화 하드코딩 텍스트
    KOREAN_TEXT = '''
옛날 옛적에 흥부와 놀부라는 형제가 살았습니다. 형 놀부는 욕심이 많고 심술궂었지만,
동생 흥부는 마음씨가 착하고 부지런하였습니다. 놀부는 부모님이 돌아가신 후 재산을 혼자
다 차지하고 흥부를 내쫓았습니다.

흥부는 가난하게 살면서도 착한 마음을 잃지 않았습니다. 어느 봄날, 흥부는 다리가 부러진
제비를 발견하였습니다. 흥부는 정성껏 제비의 다리를 고쳐 주었습니다. 가을이 되자 제비는
따뜻한 남쪽 나라로 떠났습니다.

이듬해 봄, 제비가 돌아와 흥부에게 박씨 하나를 물어다 주었습니다. 흥부는 그 박씨를
심었습니다. 박이 무럭무럭 자라 커다란 박이 열렸습니다. 흥부가 박을 타자 안에서 금은보화가
쏟아져 나왔습니다. 흥부는 부자가 되었습니다.

이 소식을 들은 놀부는 욕심이 생겼습니다. 놀부는 일부러 제비의 다리를 부러뜨렸다가
고쳐 주었습니다. 이듬해 봄, 놀부에게도 제비가 박씨를 가져다 주었습니다. 놀부가 박을
타자 안에서 도깨비들이 나와 놀부를 혼내 주었습니다. 놀부는 잘못을 뉘우쳤고, 흥부는
형을 용서하고 함께 행복하게 살았습니다.

---

옛날에 콩쥐와 팥쥐라는 자매가 있었습니다. 친엄마를 잃은 콩쥐는 계모와 의붓언니 팥쥐와
함께 살았습니다. 계모는 콩쥐를 구박하고 힘든 일만 시켰습니다.

어느 날 마을에 잔치가 열렸습니다. 팥쥐와 계모는 예쁘게 차려입고 잔치에 갔지만,
콩쥐에게는 무거운 일감을 잔뜩 주고 다 끝내야만 갈 수 있다고 하였습니다.

콩쥐가 울고 있을 때 선녀가 나타나 일을 도와주었습니다. 선녀 덕분에 일을 모두 마친
콩쥐는 예쁜 옷을 입고 잔치에 갔습니다. 잔치에서 콩쥐는 원님의 눈에 띄었습니다.
서둘러 돌아오던 콩쥐는 꽃신 한 짝을 잃어버렸습니다.

원님은 꽃신을 들고 주인을 찾았습니다. 콩쥐의 발에 꽃신이 꼭 맞았습니다. 원님은
콩쥐를 아내로 맞이하였고, 둘은 행복하게 살았습니다.

---

옛날 옛적에 토끼와 거북이가 살았습니다. 토끼는 언제나 자신의 빠른 발을 자랑하였습니다.
거북이는 느리지만 꾸준히 자신의 길을 걸었습니다.

어느 날 토끼가 거북이에게 말하였습니다.
"거북아, 나와 달리기 시합을 하자. 넌 항상 느리잖아."
거북이는 조용히 대답하였습니다.
"좋아, 한번 해 보자."

시합이 시작되었습니다. 토끼는 빠르게 달려 나갔습니다. 한참 앞서가던 토끼는 나무 그늘
아래서 낮잠을 자기로 하였습니다.
"거북이가 저렇게 느린데, 좀 쉬어도 되겠지."

거북이는 쉬지 않고 꾸준히 걸었습니다. 결국 거북이가 먼저 결승점에 도착하였습니다.
뒤늦게 달려온 토끼는 이미 늦고 말았습니다. 토끼는 자만심 때문에 진 것을 깨달았습니다.
꾸준함이 빠름을 이긴다는 것을 모두 배웠습니다.

---

옛날에 나무꾼이 산에서 나무를 하다가 도끼를 연못에 빠뜨렸습니다. 나무꾼이 슬피 울고
있을 때 산신령이 나타났습니다.

"왜 우느냐?"
"도끼를 연못에 빠뜨렸습니다."

산신령은 금도끼를 가지고 올라왔습니다.
"이것이 네 도끼냐?"
"아닙니다, 제 도끼는 낡은 쇠도끼입니다."

산신령은 다시 은도끼를 가지고 올라왔습니다.
"이것이 네 도끼냐?"
"아닙니다, 제 도끼가 아닙니다."

마지막으로 산신령은 낡은 쇠도끼를 가지고 올라왔습니다.
"이것이 네 도끼냐?"
"예, 바로 제 도끼입니다!"

산신령은 나무꾼의 정직함을 칭찬하며 금도끼, 은도끼, 쇠도끼를 모두 주었습니다.

---

옛날에 효성이 지극한 나무꾼이 살았습니다. 어머니가 병이 들자 나무꾼은 산신령을 찾아가
도움을 구하였습니다. 산신령은 깊은 산 속 약초를 알려 주었습니다.

나무꾼은 밤낮으로 걸어 약초를 구하였습니다. 어머니는 약초를 드시고 건강을 되찾으셨습니다.
마을 사람들은 나무꾼의 효심을 칭찬하였습니다.

---

아주 먼 옛날에 하늘과 땅이 처음 만들어졌습니다. 환인의 아들 환웅은 인간 세상에 내려와
살고 싶었습니다. 환인은 아들의 뜻을 알고 태백산 신단수 아래에 내려가 세상을 다스리게
하였습니다.

환웅은 바람, 비, 구름을 다스리는 신하들과 함께 인간 세상에 내려왔습니다.
그때 곰 한 마리와 호랑이 한 마리가 환웅을 찾아와 사람이 되고 싶다고 하였습니다.
환웅은 쑥과 마늘을 주며 말하였습니다.
"이것을 먹으며 백 일 동안 햇빛을 보지 않으면 사람이 될 것이다."

호랑이는 참지 못하고 동굴 밖으로 나갔습니다. 그러나 곰은 인내하며 버텼습니다.
삼칠일이 지나 곰은 아름다운 여자로 변하였습니다. 웅녀가 된 그녀는 환웅과 혼인하여
단군을 낳았습니다. 단군은 고조선을 세웠습니다.

---

옛날에 해님과 달님이 된 오누이가 살았습니다. 어머니가 떡을 팔고 돌아오다 호랑이를
만났습니다. 호랑이는 떡을 빼앗아 먹더니 어머니마저 잡아먹었습니다.

호랑이는 어머니로 변장하여 아이들이 사는 집을 찾아갔습니다.
"얘들아, 엄마 왔다. 문 열어 다오."
오누이는 이상한 낌새를 느끼고 문을 열어 주지 않았습니다.

호랑이가 안으로 들어오자 오누이는 뒷문으로 달아나 나무 위로 올라갔습니다. 호랑이도
나무 위로 올라오려 하자 오누이는 하늘에 빌었습니다.
"하늘이시여, 저희를 살려 주시려면 새 동아줄을 내려 주시고, 죽이시려면 썩은 동아줄을
내려 주소서."

하늘에서 새 동아줄이 내려왔습니다. 오누이는 줄을 잡고 하늘로 올라갔습니다.
오빠는 해님이 되고 누이는 달님이 되었습니다.

---

한국에는 아름다운 자연이 있습니다. 봄에는 진달래와 벚꽃이 피고, 여름에는 초록 산과
맑은 강이 있습니다. 가을에는 단풍이 물들고, 겨울에는 하얀 눈이 내립니다.

한국 사람들은 예로부터 자연을 사랑하고 함께 살아왔습니다. 산과 강, 바다가 어우러진
아름다운 나라입니다.

한국의 음식으로는 김치, 불고기, 비빔밥, 된장찌개, 삼겹살 등이 있습니다. 김치는 배추나
무를 소금에 절여 양념하여 발효시킨 음식입니다. 비빔밥은 밥 위에 여러 가지 나물과
고기, 달걀을 올려 고추장과 함께 비벼 먹는 음식입니다.

한국의 전통 명절로는 설날과 추석이 있습니다. 설날에는 가족이 모여 차례를 지내고
세배를 합니다. 추석에는 강강술래를 하고 송편을 만들어 먹습니다.

한국의 전통 의상은 한복입니다. 한복은 색깔이 아름답고 형태가 우아합니다.
한국의 전통 음악은 국악이라 하며, 가야금, 거문고, 해금 등의 악기를 사용합니다.

---

세종대왕은 조선의 네 번째 왕으로 1397년에 태어났습니다. 세종대왕은 백성을 사랑하는
마음이 깊었습니다. 당시 우리말을 표현할 고유한 글자가 없어 백성들이 어려움을 겪었습니다.

세종대왕은 집현전 학자들과 함께 훈민정음을 만들었습니다. 훈민정음은 1443년에 완성되었습니다.
훈민정음은 오늘날 한글로 불리며 우리 민족의 자랑스러운 문화유산입니다.

한글은 소리를 기호로 나타낸 과학적인 문자입니다. 자음 14개와 모음 10개로 이루어져 있으며,
이를 결합하여 수천 가지 음절을 표현할 수 있습니다. 한글은 배우기 쉽고 쓰기 편리하여
세계에서 가장 우수한 문자 중 하나로 인정받고 있습니다.

---

이순신 장군은 조선 시대의 위대한 장군입니다. 이순신 장군은 1545년에 태어났습니다.
임진왜란 때 이순신 장군은 거북선을 이용하여 왜군을 물리쳤습니다. 거북선은 세계 최초의
철갑선으로 알려져 있습니다.

이순신 장군은 명량 해전, 한산도 대첩 등 수많은 전투에서 승리하였습니다.
나라를 구한 이순신 장군은 오늘날에도 많은 사람들에게 존경받고 있습니다.

---

옛날에 심청이라는 효녀가 살았습니다. 심청의 아버지 심봉사는 눈이 먼 장님이었습니다.
심청은 아버지의 눈을 뜨게 하기 위해 공양미 삼백 석에 자신의 몸을 팔았습니다.

심청은 배를 타고 인당수에 몸을 던졌습니다. 옥황상제는 심청의 효심에 감동하여 심청을
연꽃 속에 태워 세상으로 돌려보냈습니다.

왕이 연꽃을 발견하여 궁으로 가져오자 연꽃이 피면서 심청이 나왔습니다. 왕은 심청을
왕비로 맞이하였습니다. 심청은 잔치를 열어 온 나라의 장님들을 초대하였습니다.
아버지 심봉사도 잔치에 와서 딸 심청을 만나는 순간 눈을 떴습니다.

---

봄이 되면 들에는 꽃들이 피어납니다. 진달래, 개나리, 벚꽃이 차례로 피어 아름다운 봄
풍경을 만들어 냅니다. 아이들은 들판에서 뛰어 놀고, 어른들은 꽃놀이를 즐깁니다.

여름에는 초록빛 나뭇잎이 무성하게 자랍니다. 더운 여름날 시원한 냇가에서 물놀이를 즐기고,
맛있는 수박을 먹습니다. 밤하늘에는 반딧불이가 반짝입니다.

가을에는 나뭇잎이 빨갛고 노랗게 물듭니다. 들판에는 황금빛 벼가 익어갑니다. 사람들은
단풍 구경을 나가고, 추석에는 가족들이 모여 차례를 지냅니다.

겨울에는 하얀 눈이 내립니다. 아이들은 눈사람을 만들고 썰매를 탑니다. 따뜻한 방에서
가족들이 모여 이야기를 나눕니다.

---

한국의 속담에는 깊은 지혜가 담겨 있습니다.

가는 말이 고와야 오는 말이 곱다는 것은 남에게 좋게 대해야 나도 좋은 대우를 받는다는
뜻입니다.

백지장도 맞들면 낫다는 것은 어떤 일이든 혼자 하는 것보다 여럿이 함께 하면 더 쉽다는
뜻입니다.

세 살 버릇 여든까지 간다는 말은 어릴 때 몸에 밴 버릇은 나이가 들어도 잘 고쳐지지
않으니 어릴 때부터 좋은 습관을 길러야 한다는 뜻입니다.

티끌 모아 태산이라는 것은 작은 것이라도 모이면 큰 것이 된다는 뜻입니다.

하늘은 스스로 돕는 자를 돕는다는 것은 자신이 노력해야 다른 도움도 의미가 있다는 뜻입니다.

---

''' * 8
    text = KOREAN_TEXT
    ckpt_prefix = 'ko'
    seed_text   = '옛날에'
    print(f'한국어 텍스트 크기: {len(text):,} 문자')
else:
    # 영어: Tiny Shakespeare 다운로드
    text = get_shakespeare()
    ckpt_prefix = 'en'
    seed_text   = 'ROMEO:'
    print(f'영어 텍스트 크기: {len(text):,} 문자')

PRETRAIN_CKPT  = f'{CHECKPOINT_DIR}/pretrain_{ckpt_prefix}_final.pt'
TOKENIZER_PATH = f'{CHECKPOINT_DIR}/tokenizer_{ckpt_prefix}.json'
LOSS_CURVE     = f'loss_curve_{ckpt_prefix}.png'

tokenizer = CharTokenizer(text)
tokenizer.save(TOKENIZER_PATH)
print(f'어휘 크기(vocab_size): {tokenizer.vocab_size}')

token_ids    = tokenizer.encode(text)
full_dataset = TextDataset(token_ids, CONTEXT_LEN)
val_size     = int(len(full_dataset) * VAL_RATIO)
train_size   = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

use_pin      = device.type == 'cuda'
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  pin_memory=use_pin)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, pin_memory=use_pin)
print(f'학습 샘플: {train_size:,}  |  검증 샘플: {val_size:,}')

# ── 모델 초기화 ──
print('\n[STEP 2] 모델 초기화 중...')
model = ToyGPT(
    vocab_size  = tokenizer.vocab_size,
    context_len = CONTEXT_LEN,
    dropout     = DROPOUT,
    **model_cfg,
).to(device)
print(f'총 파라미터 수: {model.count_parameters():,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# ── 학습 루프 ──
print('\n[STEP 3] 학습 시작!')
print('-' * 60)

def evaluate_loss(mdl, loader, dev):
    mdl.eval()
    total, cnt = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(dev), y.to(dev)
            _, loss = mdl(x, targets=y)
            total += loss.item(); cnt += 1
    mdl.train()
    return total / cnt if cnt > 0 else float('inf')

train_losses, val_losses = [], []
start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_loss, n_batches = 0.0, 0
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        _, loss = model(x, targets=y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item(); n_batches += 1
        print(f'\rEpoch {epoch}/{MAX_EPOCHS} | Batch {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}', end='', flush=True)

    avg_train = epoch_loss / n_batches
    avg_val   = evaluate_loss(model, val_loader, device)
    elapsed   = time.time() - start_time
    train_losses.append(avg_train); val_losses.append(avg_val)
    print(f'\nEpoch {epoch}/{MAX_EPOCHS} 완료 | 학습 손실: {avg_train:.4f} | 검증 손실: {avg_val:.4f} | 경과: {elapsed:.1f}s')

    if device.type == 'cuda':
        mem = torch.cuda.memory_allocated() / 1e6
        print(f'  GPU 메모리: {mem:.1f} MB')

    # 샘플 생성
    model.eval()
    with torch.no_grad():
        ids = tokenizer.encode(seed_text)
        idx = torch.tensor([ids], dtype=torch.long, device=device)
        out = model.generate(idx, max_new_tokens=150, temperature=0.8, top_k=40)
    sample = tokenizer.decode(out[0].tolist())
    print(f'\n--- 생성 샘플 (Epoch {epoch}) ---')
    print(sample[:250])
    print('-' * 60)
    model.train()

total_time = time.time() - start_time
print(f'\n총 학습 시간: {total_time:.1f}초 ({total_time/60:.1f}분)')

# ── 손실 그래프 ──
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, MAX_EPOCHS+1), train_losses, 'b-o', label='학습 손실')
ax.plot(range(1, MAX_EPOCHS+1), val_losses,   'r-o', label='검증 손실')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title(f'사전 학습 손실 곡선 ({LANGUAGE})')
ax.legend(); plt.tight_layout()
plt.savefig(LOSS_CURVE, dpi=120)
plt.show()
plt.close()

# ── 체크포인트 저장 ──
torch.save({
    'model_state' : model.state_dict(),
    'config'      : {'vocab_size': tokenizer.vocab_size, 'context_len': CONTEXT_LEN, 'dropout': DROPOUT, **model_cfg},
    'train_losses': train_losses,
    'val_losses'  : val_losses,
    'epoch'       : MAX_EPOCHS,
}, PRETRAIN_CKPT)
print(f'\n체크포인트 저장됨: {PRETRAIN_CKPT}')
print('다음 단계: Section 4 (파인튜닝) 셀을 실행하세요!')

## Section 4: 파인튜닝 (Fine-tuning)

사전 학습된 모델에 Q&A 형식을 주입합니다.

| 항목 | 사전 학습 | 파인튜닝 |
|------|---------|--------|
| 데이터 | 수십만 문자 | 25쌍 Q&A |
| 학습률 | 3e-4 | 1e-4 (낮게 — 기존 지식 보존) |
| Epoch | 5 | 30~40 (소량 데이터) |

**Before/After 비교**: 파인튜닝 전후의 응답 차이를 확인합니다.

In [ ]:
import os, time, torch
from torch.utils.data import DataLoader
from tokenizer import CharTokenizer
from model import ToyGPT
from data import TextDataset

torch.manual_seed(42)

# ── 파인튜닝 설정 ──
FT_LR         = 1e-4
FT_EPOCHS_EN  = 30
FT_EPOCHS_KO  = 40
FT_BATCH      = 8

ckpt_prefix    = 'ko' if IS_KOREAN else 'en'
PRETRAIN_CKPT  = f'{CHECKPOINT_DIR}/pretrain_{ckpt_prefix}_final.pt'
TOKENIZER_PATH = f'{CHECKPOINT_DIR}/tokenizer_{ckpt_prefix}.json'
FINETUNE_CKPT  = f'{CHECKPOINT_DIR}/finetune_{ckpt_prefix}_final.pt'

assert os.path.exists(PRETRAIN_CKPT), f'오류: {PRETRAIN_CKPT} 없음. Section 3을 먼저 실행하세요!'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Q&A 데이터 ──
EN_QA_PAIRS = [('Who wrote Romeo and Juliet?', 'William Shakespeare wrote Romeo and Juliet around 1594.'), ('What is the famous line from Hamlet?', 'To be, or not to be, that is the question.'), ("Who is Juliet's lover?", "Juliet's lover is Romeo, a young man from the Montague family."), ('What family does Romeo belong to?', 'Romeo belongs to the Montague family, rivals of the Capulets.'), ('What is the setting of Romeo and Juliet?', 'Romeo and Juliet is set in Verona, Italy.'), ("Who is Hamlet's father?", "Hamlet's father is King Hamlet, who was murdered by his brother Claudius."), ("What is Hamlet's tragic flaw?", "Hamlet's tragic flaw is his indecisiveness and inability to act quickly."), ('Who does Othello kill?', 'Othello kills his wife Desdemona after being manipulated by Iago.'), ('What is the main theme of Macbeth?', 'The main themes of Macbeth are ambition, guilt, and the corrupting power of unchecked desire.'), ('Who are the three witches in Macbeth?', "The three witches in Macbeth are supernatural beings who prophesy Macbeth's rise and fall."), ('What does Shylock demand in The Merchant of Venice?', 'Shylock demands a pound of flesh as payment for his loan to Antonio.'), ('Who is Prospero in The Tempest?', 'Prospero is the rightful Duke of Milan who uses magic on an enchanted island.'), ('What is a Shakespearean sonnet?', 'A Shakespearean sonnet has 14 lines: three quatrains and a final couplet, with rhyme scheme ABAB CDCD EFEF GG.'), ('How many plays did Shakespeare write?', 'Shakespeare wrote approximately 37 plays, including tragedies, comedies, and histories.'), ("Who is King Lear's faithful daughter?", "Cordelia is King Lear's faithful and loving daughter who refuses to flatter him falsely."), ('What causes the tragedy in Romeo and Juliet?', 'The tragedy is caused by the long feud between the Montague and Capulet families.'), ("Who is Puck in A Midsummer Night's Dream?", 'Puck is a mischievous fairy and servant to Oberon who causes comic confusion with a love potion.'), ("What is the meaning of 'All the world's a stage'?", "All the world's a stage means that life is like a play, and people are merely actors playing their parts."), ('Who kills Macbeth?', "Macduff kills Macbeth in the final battle, fulfilling the witches' prophecy."), ('What is the Globe Theatre?', "The Globe Theatre was Shakespeare's famous theatre in London, built in 1599."), ('Who is Iago in Othello?', 'Iago is the villain of Othello, a scheming soldier who manipulates everyone around him.'), ('What language did Shakespeare write in?', "Shakespeare wrote in Early Modern English, which differs from today's English but is still readable."), ('Who is Falstaff?', 'Falstaff is a comic character in Henry IV and The Merry Wives of Windsor, known for his wit and cowardice.'), ('What is a soliloquy?', 'A soliloquy is a speech where a character speaks their thoughts aloud when alone on stage.'), ('Why is Shakespeare important?', 'Shakespeare is important because he profoundly influenced the English language and literature, creating timeless stories of human nature.')]

KO_QA_PAIRS = [('흥부와 놀부 이야기에서 흥부는 어떤 사람인가요?', '흥부는 마음씨가 착하고 부지런한 사람입니다. 가난하게 살면서도 착한 마음을 잃지 않았습니다.'), ('흥부가 부자가 된 이유는 무엇인가요?', '흥부가 다리 부러진 제비를 고쳐 주었고, 이듬해 봄 제비가 박씨를 물어다 주었습니다. 박 안에서 금은보화가 나왔습니다.'), ('놀부는 왜 벌을 받았나요?', '놀부는 욕심에 일부러 제비 다리를 부러뜨렸습니다. 그 결과 박 안에서 도깨비들이 나와 놀부를 혼내 주었습니다.'), ('토끼와 거북이 이야기의 교훈은 무엇인가요?', '꾸준함이 빠름을 이긴다는 교훈입니다. 자만하지 말고 끝까지 최선을 다해야 합니다.'), ('금도끼 은도끼 이야기에서 나무꾼은 왜 상을 받았나요?', '나무꾼이 금도끼와 은도끼를 자신의 것이 아니라고 말한 정직함 때문입니다. 산신령이 정직함을 칭찬하여 세 도끼를 모두 주었습니다.'), ('심청이는 왜 인당수에 몸을 던졌나요?', '눈 먼 아버지의 눈을 뜨게 하기 위해 공양미 삼백 석에 자신의 몸을 팔았습니다. 아버지를 위한 지극한 효심 때문입니다.'), ('콩쥐팥쥐 이야기에서 콩쥐는 어떤 어려움을 겪었나요?', '친엄마를 잃고 계모와 의붓언니 팥쥐와 살았습니다. 계모는 콩쥐를 구박하고 힘든 일만 시켰습니다.'), ('단군 이야기에서 웅녀는 어떻게 사람이 되었나요?', '환웅이 준 쑥과 마늘을 먹으며 백 일 동안 햇빛을 보지 않아야 했습니다. 곰이 인내하며 버텨 아름다운 여자 웅녀가 되었습니다.'), ('한글을 만든 사람은 누구인가요?', '한글은 조선 시대 세종대왕이 집현전 학자들과 함께 만들었습니다. 1443년에 훈민정음으로 완성되었습니다.'), ('세종대왕은 왜 한글을 만들었나요?', '당시 우리말을 표현할 고유한 글자가 없어 백성들이 어려움을 겪었습니다. 세종대왕은 백성을 사랑하는 마음으로 한글을 만들었습니다.'), ('이순신 장군은 어떤 업적을 남겼나요?', '임진왜란 때 거북선을 이용하여 왜군을 물리쳤습니다. 명량 해전, 한산도 대첩 등 수많은 전투에서 승리하여 나라를 구했습니다.'), ('거북선은 무엇인가요?', '이순신 장군이 만든 배로 세계 최초의 철갑선으로 알려져 있습니다. 임진왜란 때 왜군을 물리치는 데 큰 역할을 하였습니다.'), ('한국의 전통 명절은 무엇이 있나요?', '설날과 추석이 대표적인 전통 명절입니다. 설날에는 세배를 하고, 추석에는 강강술래를 하며 송편을 만들어 먹습니다.'), ('한복은 무엇인가요?', '한복은 한국의 전통 의상입니다. 색깔이 아름답고 형태가 우아하여 명절이나 특별한 날에 입습니다.'), ('김치는 어떤 음식인가요?', '김치는 배추나 무를 소금에 절여 양념하여 발효시킨 한국 전통 음식입니다. 한국인이 가장 즐겨 먹는 음식 중 하나입니다.'), ('비빔밥은 어떻게 만드나요?', '밥 위에 여러 가지 나물과 고기, 달걀을 올려 고추장과 함께 비벼 먹는 음식입니다. 영양이 풍부하고 맛이 좋습니다.'), ('가는 말이 고와야 오는 말이 곱다는 속담은 무슨 뜻인가요?', '남에게 좋게 대해야 나도 좋은 대우를 받는다는 뜻입니다. 말과 행동을 바르게 해야 한다는 교훈을 담고 있습니다.'), ('백지장도 맞들면 낫다는 속담의 의미는 무엇인가요?', '어떤 일이든 혼자 하는 것보다 여럿이 함께 하면 더 쉽다는 뜻입니다. 협력의 중요성을 강조하는 속담입니다.'), ('티끌 모아 태산이라는 말은 무슨 뜻인가요?', '작은 것이라도 꾸준히 모으면 큰 것이 된다는 뜻입니다. 작은 노력도 소홀히 하지 말라는 교훈입니다.'), ('세 살 버릇 여든까지 간다는 속담의 교훈은 무엇인가요?', '어릴 때 몸에 밴 버릇은 나이가 들어도 잘 고쳐지지 않습니다. 어릴 때부터 좋은 습관을 길러야 한다는 뜻입니다.'), ('한국의 봄 풍경은 어떤가요?', '진달래, 개나리, 벚꽃이 차례로 피어 아름다운 봄 풍경을 만들어 냅니다. 아이들은 들판에서 뛰어 놀고 어른들은 꽃놀이를 즐깁니다.'), ('한국의 가을 풍경을 설명해 주세요.', '나뭇잎이 빨갛고 노랗게 물듭니다. 들판에는 황금빛 벼가 익어가고, 사람들은 단풍 구경을 나갑니다.'), ('단군 신화에서 고조선은 누가 세웠나요?', '웅녀와 환웅의 아들 단군이 고조선을 세웠습니다. 단군은 우리 민족의 시조로 여겨집니다.'), ('해님과 달님이 된 오누이 이야기에서 오누이는 어떻게 위기를 벗어났나요?', '호랑이가 쫓아오자 하늘에 기도하였습니다. 하늘에서 새 동아줄이 내려와 오누이는 하늘로 올라갔습니다. 오빠는 해님, 누이는 달님이 되었습니다.'), ('한국의 전통 음악을 무엇이라고 부르나요?', '한국의 전통 음악은 국악이라고 합니다. 가야금, 거문고, 해금 등의 악기를 사용하여 연주합니다.')]

QA_PAIRS = KO_QA_PAIRS if IS_KOREAN else EN_QA_PAIRS
MAX_EPOCHS_FT = FT_EPOCHS_KO if IS_KOREAN else FT_EPOCHS_EN

def build_finetune_text(qa_pairs):
    return ''.join(f'\nQ: {q}\nA: {a}\n' for q, a in qa_pairs)

# ── 모델 로드 ──
print('[STEP 1] 사전 학습 모델 로드 중...')
checkpoint = torch.load(PRETRAIN_CKPT, map_location=device, weights_only=True)
tokenizer  = CharTokenizer.load(TOKENIZER_PATH)
config     = checkpoint['config']
model = ToyGPT(**config).to(device)
model.load_state_dict(checkpoint['model_state'])
print(f'파라미터: {model.count_parameters():,}  |  사전 학습 최종 손실: {checkpoint["train_losses"][-1]:.4f}')

# ── 파인튜닝 데이터 ──
print('\n[STEP 2] 파인튜닝 데이터 준비 중...')
ft_text = build_finetune_text(QA_PAIRS)
ft_text = ''.join(ch for ch in ft_text if ch in tokenizer.vocab)  # 미지 문자 제거
token_ids   = tokenizer.encode(ft_text)
context_len = config['context_len']
dataset     = TextDataset(token_ids, context_len)
use_pin     = device.type == 'cuda'
loader      = DataLoader(dataset, batch_size=FT_BATCH, shuffle=True, pin_memory=use_pin)
print(f'파인튜닝 텍스트: {len(ft_text):,} 문자  |  Q&A 쌍: {len(QA_PAIRS)}')

# ── Before 응답 저장 ──
test_prompt = '\nQ: 흥부는 어떤 사람인가요?\nA:' if IS_KOREAN else '\nQ: Who is Romeo?\nA:'
seed_ids    = tokenizer.encode(test_prompt)
seed_tensor = torch.tensor([seed_ids], dtype=torch.long, device=device)
model.eval()
with torch.no_grad():
    before_ids  = model.generate(seed_tensor, max_new_tokens=100, temperature=0.8, top_k=40)
before_text = tokenizer.decode(before_ids[0].tolist())
print(f'\n[Before] {before_text[len(test_prompt):][:120]}')

# ── 파인튜닝 학습 루프 ──
print(f'\n[STEP 3] 파인튜닝 시작! ({MAX_EPOCHS_FT} epochs)')
print('-' * 60)

optimizer    = torch.optim.AdamW(model.parameters(), lr=FT_LR)
train_losses = []
start_time   = time.time()
sample_interval = 10 if IS_KOREAN else 5

model.train()
for epoch in range(1, MAX_EPOCHS_FT + 1):
    epoch_loss, n_batches = 0.0, 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        _, loss = model(x, targets=y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item(); n_batches += 1
        print(f'\rEpoch {epoch}/{MAX_EPOCHS_FT} | Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}', end='', flush=True)

    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    elapsed = time.time() - start_time
    print(f'\nEpoch {epoch}/{MAX_EPOCHS_FT} | 평균 손실: {avg_loss:.4f} | 경과: {elapsed:.1f}s')

    if device.type == 'cuda':
        print(f'  GPU 메모리: {torch.cuda.memory_allocated()/1e6:.1f} MB')

    if epoch % sample_interval == 0:
        model.eval()
        with torch.no_grad():
            s_ids = model.generate(seed_tensor, max_new_tokens=100, temperature=0.8, top_k=40)
        s_text = tokenizer.decode(s_ids[0].tolist())
        print(f'  샘플: {s_text[len(test_prompt):][:100]}')
        model.train()

# ── Before / After 비교 ──
print('\n' + '=' * 60)
print('  [Before vs After 비교]')
print('=' * 60)
model.eval()
with torch.no_grad():
    after_ids  = model.generate(seed_tensor, max_new_tokens=150, temperature=0.8, top_k=40)
after_text = tokenizer.decode(after_ids[0].tolist())
print(f'프롬프트: {repr(test_prompt)}\n')
print('[ Before 파인튜닝 ]')
print(before_text[len(test_prompt):][:200])
print()
print('[ After 파인튜닝 ]')
print(after_text[len(test_prompt):][:200])
print('=' * 60)

# ── 저장 ──
torch.save({'model_state': model.state_dict(), 'config': config,
            'train_losses': train_losses, 'epoch': MAX_EPOCHS_FT}, FINETUNE_CKPT)
print(f'\n체크포인트 저장됨: {FINETUNE_CKPT}')
print('다음 단계: Section 5 (대화) 셀을 실행하세요!')

## Section 5: 대화 (Chat)

**핵심 개념**: 자기회귀 생성(Autoregressive Generation)
- 매 반복마다 1회의 forward pass → 토큰 1개 생성
- 100 토큰 생성 = forward pass 100회
- 이것이 ChatGPT, Claude가 동작하는 실제 방식입니다!

아래 **모델 로드 셀**을 먼저 실행한 후,  
**대화 셀**의 `user_input`을 바꿔가며 반복 실행하세요.

In [ ]:
import torch, os, time
from tokenizer import CharTokenizer
from model import ToyGPT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_prefix    = 'ko' if IS_KOREAN else 'en'
FINETUNE_CKPT  = f'{CHECKPOINT_DIR}/finetune_{ckpt_prefix}_final.pt'
PRETRAIN_CKPT  = f'{CHECKPOINT_DIR}/pretrain_{ckpt_prefix}_final.pt'
TOKENIZER_PATH = f'{CHECKPOINT_DIR}/tokenizer_{ckpt_prefix}.json'

def load_model(path):
    assert os.path.exists(path), f'오류: {path} 없음. Section 3~4를 먼저 실행하세요!'
    ckpt  = torch.load(path, map_location=device, weights_only=True)
    mdl   = ToyGPT(**ckpt['config']).to(device)
    mdl.load_state_dict(ckpt['model_state'])
    mdl.eval()
    return mdl

print('모델 로드 중...')
tokenizer      = CharTokenizer.load(TOKENIZER_PATH)
finetune_model = load_model(FINETUNE_CKPT)
pretrain_model = load_model(PRETRAIN_CKPT)
current_model  = finetune_model

print(f"파인튜닝 모델 로드 완료  |  파라미터: {finetune_model.count_parameters():,}")
print(f"어휘 크기: {tokenizer.vocab_size}  |  디바이스: {device}")
print("\n대화 셀을 실행하여 질문하세요!")

In [ ]:
# ── 이 셀을 반복 실행하며 질문을 바꿔 보세요! ──────────────────

user_input  = "흥부와 놀부에서 흥부는 어떤 사람인가요?"  # @param {type:"string"}
temperature = 0.8  # @param {type:"slider", min:0.1, max:2.0, step:0.1}
use_model   = "파인튜닝 모델"  # @param ["파인튜닝 모델", "사전 학습 모델 (비교용)"]

STOP_SEQUENCE  = '\nQ:'
MAX_NEW_TOKENS = 200
DEFAULT_TOP_K  = 40

current_model = finetune_model if use_model.startswith('파인튜닝') else pretrain_model
model_label   = use_model

prompt = f'\nQ: {user_input}\nA:'

# ── stream_generate: 핵심 강의 포인트! ──────────────────────
# 이 루프가 GPT의 실제 동작 방식:
#   매 반복 = forward pass 1회 + 토큰 1개 샘플링
# 100 토큰 생성 = forward pass 100회 → 왜 느린지 이해!
# ─────────────────────────────────────────────────────────────
print(f'You: {user_input}')
print(f'AI ({model_label}): ', end='', flush=True)

current_model.eval()
t_start = time.time()

with torch.no_grad():
    ids     = tokenizer.encode(prompt)
    context = torch.tensor([ids], dtype=torch.long, device=device)
    gen_text = ''
    n_tokens = 0

    for _ in range(MAX_NEW_TOKENS):
        ctx_crop    = context[:, -current_model.context_len:]  # context 초과 시 crop
        logits, _   = current_model(ctx_crop)                  # forward pass 1회
        next_logits = logits[0, -1, :] / temperature           # 마지막 위치 logits

        # Top-k 필터링
        top_vals, _ = torch.topk(next_logits, min(DEFAULT_TOP_K, next_logits.size(-1)))
        next_logits = next_logits.masked_fill(next_logits < top_vals[-1], float('-inf'))

        # 확률 분포에서 샘플링
        probs      = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).unsqueeze(0)  # (1,1)

        # 실시간 스트리밍 출력
        char = tokenizer.decode([next_token.item()])
        print(char, end='', flush=True)
        gen_text += char
        n_tokens += 1

        context = torch.cat([context, next_token], dim=1)  # 새 토큰 추가

        if gen_text.endswith(STOP_SEQUENCE):  # 다음 질문 시작 시 중단
            break

elapsed     = time.time() - t_start
tok_per_sec = n_tokens / elapsed if elapsed > 0 else 0
print(f'\n[{n_tokens} 토큰 | {tok_per_sec:.1f} tok/s | {elapsed:.2f}초]')

### (보너스) 사전 학습 모델 vs 파인튜닝 모델 비교

위 대화 셀에서 `use_model` 파라미터를 변경하여 두 모델의 응답을 비교할 수 있습니다.

- **사전 학습 모델**: Q: 다음에 랜덤한 텍스트 생성 (Q&A 패턴 미학습)
- **파인튜닝 모델**: Q: 다음에 A: 로 답하는 패턴 학습됨